In [3]:
import httpx
import pandas as pd
import numpy as np
import psycopg
from datetime import datetime, timedelta, timezone

import os
import sys
from dotenv import load_dotenv
sys.path.append(os.path.abspath('./src'))
sys.path.append(os.path.abspath('./src/logger'))

import db_functions as dbf

import logging
import basic_logger
logger = basic_logger.initiate_basic_logger()

C:\Users\benib\Projects\Dota\logs


In [4]:
db = dbf.DotaDB()
query = """
            query($id: Long!) {
                match(id: $id) {
                    id
                    tournamentId
                    tournamentRound
                    leagueId
                    radiantTeamId
                    direTeamId
                    seriesId
                    gameVersionId
                    regionId
                    clusterId
                    didRadiantWin
                    startDateTime
                    endDateTime
                    durationSeconds
                    firstBloodTime
                    towerStatusRadiant
                    towerStatusDire
                    barracksStatusRadiant
                    barracksStatusDire
                    rank
                    actualRank
                    averageRank
                    averageImp
                    bracket
                    analysisOutcome
                    topLaneOutcome
                    midLaneOutcome
                    bottomLaneOutcome
                    predictedOutcomeWeight
                    pickBans {
                        isPick
                        heroId
                        order
                        isRadiant
                    }
                    chatEvents {
                        time
                        type
                        fromHeroId
                        toHeroId
                        value
                        pausedTick
                        isRadiant
                    }
                    predictedWinRates
                    winRates
                    radiantNetworthLeads
                    radiantExperienceLeads
                    radiantKills
                    direKills
                    towerDeaths {
                        time
                        npcId
                        isRadiant
                        attacker
                    }
                    towerStatus {
                        towers {
                            npcId
                            hp
                        }
                    outposts {
                        npcId
                        isControlledByRadiant
                        isRadiantSide
                    }
                }
                players {
                heroId
                steamAccountId
                partyId
                steamAccount {
                    name
                    realName
                    profileUri
                    timeCreated
                    isAnonymous
                    proSteamAccount {
                        teamId
                        name
                    }
                }
                isRadiant
                isVictory
                variant
                imp
                lane
                position
                networth
                goldPerMinute
                goldSpent
                towerDamage
                heroDamage
                intentionalFeeding
                stats {
                    impPerMinute
                    goldPerMinute
                    networthPerMinute
                    experiencePerMinute
                    towerDamagePerMinute
                    campStack
                    deathEvents {
                    time
                    attacker
                    isDieBack
                    }
                    farmDistributionReport {
                    creepLocation {
                        id
                        gold
                    }
                    neutralLocation {
                        id
                        gold
                    }
                    ancientLocation {
                        id
                        gold
                    }
                    buildings {
                        id
                        gold
                    }
                    bountyGold {
                        id
                        gold
                    }
                    other {
                        id
                        gold
                    }
                    buyBackGold
                    }
                    matchPlayerBuffEvent {
                    time
                    abilityId
                    itemId
                    stackCount
                    }
                    inventoryReport {
                    item0 {
                        itemId
                    }
                    item1 {
                        itemId
                    }
                    item2 {
                        itemId
                    }
                    item3 {
                        itemId
                    }
                    item4 {
                        itemId
                    }
                    item5 {
                        itemId
                    }
                    neutral0 {
                        itemId
                    }
                    }
                    itemPurchases {
                    time
                    itemId
                    }
                    courierKills {
                    time
                    }
                    runes {
                    time
                    rune
                    action
                    positionX
                    positionY
                    }
                    wards {
                    time
                    type
                    positionX
                    positionY
                    }
                    wardDestruction {
                    time
                    gold
                    isWard
                    }
                }
                }
            }
            }
        """

In [81]:
results = db.query_stratz(query, variables={'id': 8183642521})
match_json = results['data']['match']

INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"


In [36]:
match_details = {}
for key, value in match_json.items():
    if type(value) != list:
        match_details[key] = value
    if type(value) == list:
        print(key, type(value[0]))

pickBans <class 'dict'>
chatEvents <class 'dict'>
predictedWinRates <class 'float'>
winRates <class 'float'>
radiantNetworthLeads <class 'int'>
radiantExperienceLeads <class 'int'>
radiantKills <class 'int'>
direKills <class 'int'>
towerDeaths <class 'dict'>
towerStatus <class 'dict'>
players <class 'dict'>


In [ ]:
snapshots = []
tower_updates = []
outpost_updates = []
for index, buildings in enumerate(match_json['towerStatus']):
    snapshot_id = str(match_json['id']) + f'_{index}'
    snapshots.append(
    {
        'snapshot_id': snapshot_id,
        'match_id': match_json['id'],
        'order_index': index
    })

    towers = buildings['towers']
    for entry in towers:
        entry['snapshot_id'] = snapshot_id
    tower_updates.append(towers)

    outposts = buildings['outposts']
    for entry in outposts:
        entry['snapshot_id'] = snapshot_id
    outpost_updates.append(outposts)
storage['snapshots'].extend(snapshots)
storage['towerStatus'].extend(tower_updates)
storage['outposts'].extend(outpost_updates)

In [87]:
len(storage['towerStatus'])

10

In [66]:
match_json['towerStatus']

[{'towers': [{'npcId': 42, 'hp': 1300},
   {'npcId': 40, 'hp': 2200},
   {'npcId': 43, 'hp': 1300},
   {'npcId': 38, 'hp': 2200},
   {'npcId': 41, 'hp': 1300},
   {'npcId': 46, 'hp': 2200},
   {'npcId': 49, 'hp': 1300},
   {'npcId': 48, 'hp': 1300},
   {'npcId': 45, 'hp': 2200},
   {'npcId': 44, 'hp': 2200},
   {'npcId': 47, 'hp': 1300},
   {'npcId': 51, 'hp': 4500},
   {'npcId': 50, 'hp': 4500},
   {'npcId': 35, 'hp': 2600},
   {'npcId': 29, 'hp': 2500},
   {'npcId': 26, 'hp': 1800},
   {'npcId': 30, 'hp': 2500},
   {'npcId': 27, 'hp': 1800},
   {'npcId': 31, 'hp': 2500},
   {'npcId': 34, 'hp': 2500},
   {'npcId': 32, 'hp': 2500},
   {'npcId': 33, 'hp': 2500},
   {'npcId': 35, 'hp': 2600},
   {'npcId': 25, 'hp': 2600},
   {'npcId': 17, 'hp': 1800},
   {'npcId': 28, 'hp': 1800},
   {'npcId': 24, 'hp': 2500},
   {'npcId': 23, 'hp': 2500},
   {'npcId': 22, 'hp': 2500},
   {'npcId': 21, 'hp': 2500},
   {'npcId': 18, 'hp': 1800},
   {'npcId': 20, 'hp': 2500},
   {'npcId': 19, 'hp': 2500},


In [42]:
match_json['towerStatus']

[{'towers': [{'npcId': 42, 'hp': 1300},
   {'npcId': 40, 'hp': 2200},
   {'npcId': 43, 'hp': 1300},
   {'npcId': 38, 'hp': 2200},
   {'npcId': 41, 'hp': 1300},
   {'npcId': 46, 'hp': 2200},
   {'npcId': 49, 'hp': 1300},
   {'npcId': 48, 'hp': 1300},
   {'npcId': 45, 'hp': 2200},
   {'npcId': 44, 'hp': 2200},
   {'npcId': 47, 'hp': 1300},
   {'npcId': 51, 'hp': 4500},
   {'npcId': 50, 'hp': 4500},
   {'npcId': 35, 'hp': 2600},
   {'npcId': 29, 'hp': 2500},
   {'npcId': 26, 'hp': 1800},
   {'npcId': 30, 'hp': 2500},
   {'npcId': 27, 'hp': 1800},
   {'npcId': 31, 'hp': 2500},
   {'npcId': 34, 'hp': 2500},
   {'npcId': 32, 'hp': 2500},
   {'npcId': 33, 'hp': 2500},
   {'npcId': 35, 'hp': 2600},
   {'npcId': 25, 'hp': 2600},
   {'npcId': 17, 'hp': 1800},
   {'npcId': 28, 'hp': 1800},
   {'npcId': 24, 'hp': 2500},
   {'npcId': 23, 'hp': 2500},
   {'npcId': 22, 'hp': 2500},
   {'npcId': 21, 'hp': 2500},
   {'npcId': 18, 'hp': 1800},
   {'npcId': 20, 'hp': 2500},
   {'npcId': 19, 'hp': 2500},


In [8]:
results

{'data': {'match': {'id': 8400978881,
   'tournamentId': None,
   'tournamentRound': None,
   'leagueId': 18359,
   'radiantTeamId': 9640842,
   'direTeamId': 8291895,
   'seriesId': 999302,
   'gameVersionId': 180,
   'regionId': 25,
   'clusterId': 232,
   'didRadiantWin': False,
   'startDateTime': 1754224445,
   'endDateTime': 1754227221,
   'durationSeconds': 2776,
   'firstBloodTime': 18,
   'towerStatusRadiant': 1536,
   'towerStatusDire': 1974,
   'barracksStatusRadiant': 0,
   'barracksStatusDire': 63,
   'rank': 80,
   'actualRank': 80,
   'averageRank': None,
   'averageImp': -5,
   'bracket': 8,
   'analysisOutcome': 'COMEBACK',
   'topLaneOutcome': 'RADIANT_VICTORY',
   'midLaneOutcome': 'RADIANT_VICTORY',
   'bottomLaneOutcome': 'RADIANT_VICTORY',
   'predictedOutcomeWeight': 38,
   'pickBans': [{'isPick': False,
     'heroId': 34,
     'order': 0,
     'isRadiant': False},
    {'isPick': False, 'heroId': 45, 'order': 1, 'isRadiant': True},
    {'isPick': False, 'heroId':

In [ ]:
db.query_matches([8400889142])

INFO: Processing 8400889142 (1/1)
INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"
Error inserting data into table 'match_details': duplicate key value violates unique constraint "match_details_pkey"
DETAIL:  Key (id)=(8400889142) already exists.
CONTEXT:  COPY match_details, line 1
INFO: Bulk inserted 1 rows into match_details
Data inserted into table 'match_pick_bans' successfully.
INFO: Bulk inserted 24 rows into match_pick_bans
Error inserting data into table 'match_players': column "steamAccount.name" of relation "match_players" does not exist
INFO: Bulk inserted 10 rows into match_players
Error inserting data into table 'match_performance_metrics': column "networth" of relation "match_performance_metrics" does not exist
INFO: Bulk inserted 370 rows into match_performance_metrics
